# 11 · Cascade: ASR → VQA (T1_Speech)

Transcripts from `asr/` go into **Maryna's `answer()`** instead of the text question.
Same VQA model, same prompt, same ids as her best T2_Text run, so the only difference is the question source.

**Before running, agree with Maryna:**
1. where `answer()` lives (below: `from vqa import answer`, change if needed);
2. `VQA_MODEL` and `PROMPT_ID` of her best T2_Text config;
3. where to run: a 7B VLM needs the Kaggle GPU. On Kaggle, upload your `asr/dev_en_*.jsonl`
   as a Kaggle dataset and copy it to `/kaggle/working/asr/` (with `MMCQA_OUT=/kaggle/working`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root: common.py, speech.py

from common import *
from speech import *

In [ ]:
from vqa import answer          # <- Maryna's answer(image_paths, questions, prompt_id)

SPLIT = "dev"
ASR_MODEL = "whisper-large-v3-turbo"
ASR_SHORT = "whisperturbo"      # short name used in the preds file name
VQA_MODEL = "qwen25vl7b"        # <- same as Maryna's best T2_Text run
PROMPT_ID = "p2"                # <- same as Maryna's best T2_Text run
BATCH = 8

In [ ]:
rows = cascade_inputs(asr_path(SPLIT, ASR_MODEL), SPLIT)
out = preds_path("t1_speech_cascade", SPLIT, f"{ASR_SHORT}+{VQA_MODEL}", PROMPT_ID)
print(len(rows), "questions ->", out)
print("text:", rows[0]["question_text"])
print("asr: ", rows[0]["question"])

## Run (resumable, appends after every batch)

In [ ]:
from tqdm.auto import tqdm

done = {r["id"] for r in load_jsonl(out)} if out.exists() else set()
todo = [r for r in rows if r["id"] not in done]
print(f"{len(done)} done, {len(todo)} to go")

for i in tqdm(range(0, len(todo), BATCH)):
    b = todo[i:i + BATCH]
    preds = answer([r["image_path"] for r in b], [r["question"] for r in b], prompt_id=PROMPT_ID)
    save_jsonl([{"id": r["id"], "lang": LANG, "split": SPLIT, "task": "t1_speech_cascade",
                 "model": f"{ASR_SHORT}+{VQA_MODEL}", "prompt_id": PROMPT_ID, "pred": p}
                for r, p in zip(b, preds)], out, append=True)

## Score

In [ ]:
res = evaluate(out, SPLIT)      # -> results/<preds name>.json + <preds>.scores.jsonl
res

Sanity: the T2_Text run you compare against must cover the same ids.

In [ ]:
text_out = preds_path("t2_text", SPLIT, VQA_MODEL, PROMPT_ID)
if text_out.exists():
    a = {r["id"] for r in load_jsonl(out)}
    b = {r["id"] for r in load_jsonl(text_out)}
    print(f"speech {len(a)}, text {len(b)}, common {len(a & b)}")
else:
    print("no T2_Text preds yet:", text_out)